# calc — a literate reading

*Produced by the `literate-repo` skill from the repository in `../sample_repo`.*

`calc` is a pocket calculator: it turns a string like `1 + 2 * 3` into the number `7.0`, honoring operator precedence, parentheses, and unary minus. It is small — about 100 lines — which lets us read **all** of it, the way Knuth read TeX and *Physically Based Rendering* reads a renderer: as an essay whose subject is a program.

## How to read this

These notebooks are meant to be read **in order**, front to back. Each introduces only ideas that earlier chapters have already established.

- **Prose leads; code follows.** Every excerpt is preceded by the idea it embodies.
- **Named fragments.** A bracketed phrase like ⟨Tokenize, parse, evaluate 1.2⟩ stands for code refined in the section whose number it carries. This lets you see the whole shape before any detail, and descend only where you wish.
- **Sections are numbered** (§1.1, §1.2, …) so fragments and prose can cross-reference.
- **Provenance.** Each excerpt is the repo's real code, headed by its `file : lines`.
- **Runnable.** Because `calc` is Python, the demonstration cells import the real package and run it — a literate reading you can execute.

## The shape of the system

`calc` is the textbook three-stage pipeline. A string becomes a list of **tokens**, the tokens become an **abstract syntax tree** (AST), and the tree is walked to a **number**:

```mermaid
flowchart LR
    A["source string<br/>\"1 + 2 * 3\""] -->|tokenize| B["tokens<br/>NUMBER + NUMBER * NUMBER"]
    B -->|parse| C["AST<br/>(+ 1 (* 2 3))"]
    C -->|evaluate| D["number<br/>7.0"]
```

Plain-text fallback, if your viewer does not render Mermaid:

```text
  "1 + 2 * 3"  --tokenize-->  [NUMBER + NUMBER * NUMBER]  --parse-->  (+ 1 (* 2 3))  --evaluate-->  7.0
```

## Reading order is not file order

On disk the modules sort roughly alphabetically — `cli`, `evaluator`, `nodes`, `parser`, `tokens`. That is the order Python's importer is happy with, not the order a person learns the system in. We read it in the order ideas *depend* on one another:

| Chapter | Notebook | Source it reads |
|---|---|---|
| §1 The system at a glance | this notebook | `cli.py` |
| §2 Reading input: the tokenizer | [01](01_reading_input_the_tokenizer.ipynb) | `tokens.py` |
| §3 Structure: the parser | [02](02_structure_the_parser.ipynb) | `nodes.py`, `parser.py` |
| §4 Meaning: the evaluator | [03](03_meaning_the_evaluator.ipynb) | `evaluator.py` |

Rearranging the source into the order best for human understanding is the whole point of a literate reading; the compiler already has its own order.

## §1 The system at a glance

The top of the program is `calc()`: one function that names the three stages and nothing else. Read it and you have the entire arc of a run; each bracketed stage is a fragment refined in a later chapter.

📄 [calc/cli.py lines 16–20](../sample_repo/calc/cli.py)

```python
# calc/cli.py : lines 16-20
def calc(text: str) -> float:
    """Evaluate a single expression string end to end."""
    ⟨Turn the string into tokens 2⟩        # tokens = tokenize(text)
    ⟨Turn the tokens into a syntax tree 3⟩  # tree = parse(tokens)
    ⟨Walk the tree to a number 4⟩           # return evaluate(tree)
```

That is the book in miniature: chapters §2, §3, and §4 refine the three fragments. Notice the staging — each function consumes exactly what the previous one produced, and the data type changes at every arrow (string → tokens → tree → number). The clean handoffs are why each stage can be understood, and tested, on its own.

### §1.1 Around the core: the command line

`calc()` is wrapped by `main()`, which only decides *where the text comes from* — an argument, or a line of standard input — and *where errors go*. Keeping this apart from `calc()` means the evaluation pipeline never has to know it is running in a terminal.

📄 [calc/cli.py lines 23–35](../sample_repo/calc/cli.py)

```python
# calc/cli.py : lines 23-35
def main(argv: list[str]) -> int:
    if len(argv) > 1:
        print(calc(" ".join(argv[1:])))
        return 0
    for line in sys.stdin:  # REPL mode: one expression per line
        line = line.strip()
        if not line:
            continue
        try:
            print(calc(line))
        except (SyntaxError, ZeroDivisionError, TypeError) as e:
            print(f"error: {e}")
    return 0
```

The `try/except` here is the program's single error boundary: the inner stages raise `SyntaxError`, `ZeroDivisionError`, or `TypeError` freely, and this one place turns them into a friendly message. We will see each stage trust that arrangement and not re-check.

## See it run

Before descending, let us confirm the whole pipeline works, end to end:

In [1]:
import os, sys
# Find the bundled demo package (works when run from the calc-literate directory).
d = os.getcwd()
for _ in range(6):
    for cand in (os.path.join(d, 'sample_repo'), os.path.join(d, '..', 'sample_repo')):
        if os.path.isdir(os.path.join(cand, 'calc')):
            sys.path.insert(0, os.path.abspath(cand))
            break
    else:
        d = os.path.dirname(d); continue
    break
import calc
print('imported calc', calc.__version__)

imported calc 0.1.0


In [2]:
for expr in ['1 + 2 * 3', '(1 + 2) * 3', '-3 + 4', '8 - 2 - 1', '2 * (3 + 4) - 1']:
    print(f'{expr:>16}  =  {calc.calc(expr)}')

       1 + 2 * 3  =  7.0
     (1 + 2) * 3  =  9.0
          -3 + 4  =  1.0
       8 - 2 - 1  =  5.0
 2 * (3 + 4) - 1  =  13.0


## Where this leads

You now hold the skeleton: tokenize → parse → evaluate, wrapped by a thin command line. The next three chapters refine the three fragments in dependency order. We begin with the simplest, the tokenizer (§2), because the parser (§3) consumes its output, and the evaluator (§4) consumes the parser's.

---

### Fragment index

| Fragment | Defined in |
|---|---|
| ⟨Turn the string into tokens⟩ | §2 |
| ⟨Turn the tokens into a syntax tree⟩ | §3 |
| ⟨Walk the tree to a number⟩ | §4 |